<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c39-header.png" alt="Nextia Learning · Entity Resolution and Record Matching: Is This the Same Thing?" width="100%">

# Solution: Embeddings for names, addresses and products

**[Embeddings for names, addresses and products](https://learning.nextia-ai.com/courses/record-matching/m06/embeddings-for-names-addresses-and-products/)** · Entity Resolution and Record Matching: Is This the Same Thing? · Module 6, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** read shipped embeddings, compute cosines, and see why embeddings do not separate the people of one household or a single item from a pack of 2.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells (about 33 MB). |
| **You should know** | Comparison vectors, from [Compare by field, then combine](https://learning.nextia-ai.com/courses/record-matching/m04/compare-by-field-then-combine/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (rapidfuzz 3.14.6, jellyfish 1.3.0, datasketch 2.0.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/record-matching/m06/embeddings-for-names-addresses-and-products/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C39/M06-L03-embeddings-for-names-addresses-and-products/embeddings-for-names-addresses-and-products-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 36 MB) into a folder `larkfield-match` and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-match` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Every customer record and every product listing is **synthetic**: our code made them from invented people and products, and then added realistic errors on purpose (typing errors, nicknames, old addresses, shared family emails). Because the code knows the true person behind each record, you can always check an answer. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C39/data/dataset-card.md) explains the error model.

In [ ]:
# Setup: download the files into larkfield-match/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C39/"
FILES = {
    "data/sources/shop_accounts.csv": "f59f3a9619c35b572650e10663a0ba5f316202d173850689ec9242706fb4174e",
    "data/sources/loyalty_members.csv": "629696f38469898be48cb664e5d69f9c6d438e8622899bbb281bd520bf44af32",
    "data/sources/garden_centres.csv": "2878063d24a38d841f93bb14e07b2d46619fcc4a3909e92ce0238116434d85b2",
    "data/truth.csv": "d951378129bdf30b649250e6d2127787e5630d66aa7cfe5793b1a110e37a06d3",
    "data/embeddings/customers_ids.csv": "1848dfc7f3b399434a8c664137df85fca194d4fb7517c5781820cb975c4421bb",
    "data/embeddings/customers_int8.npy": "84b7a1b69596ed06710e602ccc63c3542747e5fe72ede2025e1e8cbb01102ae9",
    "data/embeddings/customers_scale.npy": "e67c0f4afac7b2d99186b375355f98e5137b3736c59d04c08e0d3e1a282dd4d1",
    "data/embeddings/products_ids.csv": "221e4977a8139371bb9757208256b83cb81d3e8d5022144003f43ba1aa0a2ffd",
    "data/embeddings/products_int8.npy": "192de529abd380a4a20680c228fba56698bd95b1a26280698f2a7b4d0a55f2b2",
    "data/embeddings/products_scale.npy": "e0dc91c65c391326cace2c364a36d420db63fc7190b301f27268990f228e1441",
    "data/embeddings/model.json": "08acf932473cd6d3a6d61072a1b26245dab3067cef2adf80205d6e9a3ade4da9",
    "data/products/feed_a.csv": "b5badaa6f0a813a19bb321263f46950100ef3a4e9a046163ed97e7af60fc94e2",
    "data/products/feed_b.csv": "c6bcc923db6b503859b20b5e4e1833d3376b08bd96b9d65c0392ae07a480e453",
    "data/products/truth.csv": "64adad1022e94e76841d3f34e339a0cc14cd1be809e2d69ab50a034d55b0b983",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-match").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-match and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-match.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd` and matplotlib. It defines a few small helpers:

- `source(name)` reads one customer source as it is (`"shop"`, `"loyalty"` or `"garden"`); `records()` reads all three in shared columns (`record_id, source, name, email, phone, address, postcode, date_of_birth`), with the values exactly as the sources wrote them.
- `truth()` says which true customer (`entity_id`) each record belongs to, and its split (`train`, `validation` or `test`).
- `pair_measures(pairs)` measures a table of pairs (`record_id_l`, `record_id_r`) that a matcher calls "the same": pair precision, recall and F1 on the test split, as in the project's `match eval`.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, pair measures, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

FILES_OF = {"shop": "shop_accounts.csv", "loyalty": "loyalty_members.csv", "garden": "garden_centres.csv"}

def source(name):
    """One customer source as it is: its own columns, every value as text."""
    return pd.read_csv(f"data/sources/{FILES_OF[name]}", dtype=str, keep_default_na=False)

def records():
    """All customer records in shared columns, values as the sources wrote them."""
    s, l, g = source("shop"), source("loyalty"), source("garden")
    parts = [
        pd.DataFrame({"record_id": s.record_id, "source": "shop", "name": s.first_name + " " + s.last_name,
                      "email": s.email, "phone": s.phone, "address": s.address + ", " + s.city + " " + s.province,
                      "postcode": s.postcode, "date_of_birth": s.date_of_birth}),
        pd.DataFrame({"record_id": l.record_id, "source": "loyalty", "name": l.full_name, "email": l.email,
                      "phone": l.phone, "address": "", "postcode": l.postcode, "date_of_birth": l.date_of_birth}),
        pd.DataFrame({"record_id": g.record_id, "source": "garden", "name": g.name, "email": g.email,
                      "phone": g.phone, "address": g.address, "postcode": "", "date_of_birth": g.date_of_birth}),
    ]
    return pd.concat(parts, ignore_index=True)

def truth():
    """record_id -> entity_id (the true customer) and split."""
    return pd.read_csv("data/truth.csv", dtype=str)

def ordered(pairs):
    """The same pairs with the smaller record ID first, one row per pair."""
    lo = pairs[["record_id_l", "record_id_r"]].min(axis=1)
    hi = pairs[["record_id_l", "record_id_r"]].max(axis=1)
    out = pd.DataFrame({"record_id_l": lo, "record_id_r": hi})
    return out[out.record_id_l != out.record_id_r].drop_duplicates()

def true_pairs():
    """Every pair of records of the same true customer, with its split."""
    t = truth()
    m = t.merge(t, on="entity_id", suffixes=("_l", "_r"))
    m = m[m.record_id_l < m.record_id_r]
    return m[["record_id_l", "record_id_r", "split_l"]].rename(columns={"split_l": "split"})

RANK = {"train": 0, "validation": 1, "test": 2}

def pair_measures(pairs, split="test"):
    """Pair precision, recall and F1 of the pairs a matcher calls the same (a pair is in the test split when one
    of its records belongs to a test customer)."""
    p = ordered(pairs)
    t = truth().set_index("record_id")
    rank = t.split.map(RANK)
    p = p.assign(split=np.maximum(p.record_id_l.map(rank), p.record_id_r.map(rank)).map({v: k for k, v in RANK.items()}),
                 same=(p.record_id_l.map(t.entity_id) == p.record_id_r.map(t.entity_id)))
    tp_all = true_pairs()
    p, tp_all = p[p.split == split], tp_all[tp_all.split == split]
    tp = int(p.same.sum())
    precision = tp / len(p) if len(p) else 0.0
    recall = tp / len(tp_all) if len(tp_all) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3),
            "pairs": len(p), "right": tp, "true_pairs": len(tp_all)}

def same_value_pairs(df, col):
    """Every pair of records that share a non-empty value of col."""
    d = df[df[col].notna() & (df[col] != "")][["record_id", col]]
    sizes = d[col].value_counts()
    n_pairs = int((sizes * (sizes - 1) // 2).sum())
    if n_pairs > 20_000_000:   # protect the computer's memory from a key that is too wide
        raise ValueError(f"This key makes {n_pairs:,} pairs: too many to hold in memory. Use a narrower key.")
    m = d.merge(d, on=col, suffixes=("_l", "_r"))
    return m[m.record_id_l < m.record_id_r][["record_id_l", "record_id_r"]].drop_duplicates()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## Setup: the clean records of Module 2

The next cell cleans every record as Module 2 did: names without accents and titles, split into first name and
surname; emails in lower case; phones as 10 digits; postcodes as 6 characters; dates as YYYY-MM-DD. Run it. It
prints the number of records.

In [ ]:
# The clean records (Module 2): one row per record with first, last, email, phone, postcode, dob.
import re, unicodedata

def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def norm_text(s):
    s = strip_accents(str(s or "")).lower().replace("'", "")
    return " ".join(w for w in re.sub(r"[^\w\s]", " ", s).split() if w not in {"mr", "mrs", "ms", "dr", "m", "mme"})

def norm_phone(s):
    d = re.sub(r"\D", "", re.split(r"(?i)\s*ext", str(s or ""))[0])
    d = d[1:] if len(d) == 11 and d.startswith("1") else d
    return d if len(d) == 10 else ""

def norm_postcode(s):
    p = re.sub(r"[\s-]", "", str(s or "")).upper()
    return p if re.fullmatch(r"[A-Z]\d[A-Z]\d[A-Z]\d", p) else ""

def norm_date(s):
    s = str(s or "")
    m = re.fullmatch(r"(\d{1,2})/(\d{1,2})/(\d{4})", s)
    return f"{m[3]}-{int(m[2]):02d}-{int(m[1]):02d}" if m else (s if re.fullmatch(r"\d{4}-\d\d-\d\d", s) else "")

def split_name(s):
    s = str(s or "")
    if "," in s:
        last, first = s.split(",", 1)
    else:
        words = s.split()
        first, last = " ".join(words[:-1]), (words[-1] if words else "")
    return norm_text(first), norm_text(last)

def postcode_of(postcode, address):
    """The postcode column, or the postcode at the end of a garden-centre address."""
    found = re.findall(r"[A-Za-z]\d[A-Za-z][ -]?\d[A-Za-z]\d", address)
    return norm_postcode(postcode) or (norm_postcode(found[-1]) if found else "")

recs = records()
names = recs.name.map(split_name)
clean = pd.DataFrame({
    "record_id": recs.record_id, "source": recs.source,
    "first": [f for f, _ in names], "last": [l.replace(" ", "") for _, l in names],
    "email": recs.email.str.strip().str.lower(), "phone": recs.phone.map(norm_phone),
    "postcode": [postcode_of(p, a) for p, a in zip(recs.postcode, recs.address)],
    "dob": recs.date_of_birth.map(norm_date),
})
print(f"{len(clean):,} clean records")

## 1. The shipped embeddings

An **embedding** is a list of numbers that a model makes from a text. Texts with a similar meaning get similar lists. The course data already has the embedding of every customer record (its name and address) and of every product title, made by a small model. So this notebook does not run the model: it only reads numbers.

Each embedding has 384 numbers. They are stored as small whole numbers (8 bits) with one scale per row, a quarter of the size of normal numbers.

In [ ]:
def load_embeddings(dataset):
    """The shipped embeddings: 8-bit numbers times one scale per row, then length 1 (so a dot product is the cosine)."""
    e = np.load(f"data/embeddings/{dataset}_int8.npy").astype("float32") * np.load(f"data/embeddings/{dataset}_scale.npy")
    e /= np.linalg.norm(e, axis=1, keepdims=True)
    ids = pd.read_csv(f"data/embeddings/{dataset}_ids.csv", dtype=str).record_id
    return e, {r: i for i, r in enumerate(ids)}

E, row = load_embeddings("customers")
print(json.load(open("data/embeddings/model.json"))["model"], E.shape)
print("the first 8 of 384 numbers of G-000001:", E[row["G-000001"], :8].round(3))

## 2. Cosine: how alike are two embeddings?

The **cosine** of two embeddings is 1 when they point the same way, and lower when they differ. Because every row has length 1, the cosine is the sum of the products of the numbers.

> **Predict.** Valérie Tremblay's two records are one customer. Samuel and Nathan Green are two brothers at one address. Which pair gets the higher cosine?

In [ ]:
def cosine(emb, rows, a, b):
    return float(emb[rows[a]] @ emb[rows[b]])

recs = records().set_index("record_id")
for a, b in [("G-000001", "S-009232"), ("G-001688", "G-002894")]:
    print(recs.loc[a, "name"], "|", recs.loc[a, "address"])
    print(recs.loc[b, "name"], "|", recs.loc[b, "address"])
    print("  cosine", round(cosine(E, row, a, b), 3), "\n")

## 3. Customers: true pairs against other pairs

The next cell takes every pair of records with the same postcode and computes its cosine. Then it compares one-customer pairs with pairs of two people.

In [ ]:
keys = clean.assign(k1=clean.postcode)
cand = same_value_pairs(keys, "k1").reset_index(drop=True)
t = truth().set_index("record_id")
cand["same"] = (cand.record_id_l.map(t.entity_id) == cand.record_id_r.map(t.entity_id)).to_numpy()
cand["cosine"] = (E[cand.record_id_l.map(row)] * E[cand.record_id_r.map(row)]).sum(axis=1)
print(cand.groupby("same").cosine.median().round(3))
plt.hist([cand[cand.same].cosine, cand[~cand.same].cosine], bins=np.arange(0.5, 1.001, 0.02), density=True,
         label=["one customer", "two people"])
plt.xlabel("cosine of the two records' embeddings"); plt.ylabel("share of pairs"); plt.legend(); plt.show()

> **Check.** Median cosine: 0.865 for one customer, 0.909 for two people at one postcode. The pairs of two people score **higher**. Two people at one postcode often share the whole address, written the same way; one customer's two records often write the address in two ways, or one of them is an old address. The address is most of the text, so it decides the cosine.

## 4. Products: true matches

Now the supplier feeds. A true match is one product in both feeds, often with a very different title.

In [ ]:
P, prow = load_embeddings("products")
fa = pd.read_csv("data/products/feed_a.csv", dtype=str, keep_default_na=False)
fb = pd.read_csv("data/products/feed_b.csv", dtype=str, keep_default_na=False)
pt = pd.read_csv("data/products/truth.csv", dtype=str)
ent = pt.set_index("product_id").product_entity
# true matches: one product in both feeds
both = fa.assign(e=fa.product_id.map(ent)).merge(fb.assign(e=fb.product_id.map(ent)), on="e", suffixes=("_a", "_b"))
true_cos = (P[both.product_id_a.map(prow)] * P[both.product_id_b.map(prow)]).sum(axis=1)
print(len(both), "true matches; median cosine", round(float(np.median(true_cos)), 3))
print(both[["title_a", "title_b"]].head(3).to_string(index=False))

## 5. Your turn: a single item against a pack of 2

Feed A sells some products as a single item and as a pack of 2, with almost the same title. They are two different products. Compute the cosine of each pack with its single item.

In [ ]:
single = fa.set_index("title").product_id
packs = fa[fa.title.str.endswith(" (Pack of 2)")]
packs = packs.assign(single_id=packs.title.str.removesuffix(" (Pack of 2)").map(single)).dropna(subset=["single_id"])
pack_cos = (P[packs.product_id.map(prow)] * P[packs.single_id.map(prow)]).sum(axis=1)
pack_median = round(float(np.median(pack_cos)), 3)
above = int((pack_cos >= np.median(true_cos)).sum())
print(len(packs), "pack-of-2 pairs; median cosine", pack_median, "; above the true matches' median:", above)

In [ ]:
expect('pack_median', pack_median, 0.989, tolerance=0.001)

In [ ]:
expect('above', above, 79)

> **Check.** The 79 pack pairs have a median cosine of 0.989, and 79 of them are at least as alike as the median true match (0.952): a single item and its pack look more alike to the model than most true matches. For the model, "(Pack of 2)" is a small change of text, not a different product.

## 6. Sizes

Run the next cell. The same planter in other sizes and colours gets a high cosine too. Numbers and units carry little weight in an embedding.

In [ ]:
# The same product line in two sizes: what does the model see?
planter = fa[fa.title.str.startswith("Mossway Ceramic Planter")].head(6)
base = planter.product_id.iloc[0]
print(planter.title.iloc[0])
for pid, title in zip(planter.product_id.iloc[1:], planter.title.iloc[1:]):
    print(f"  {float(P[prow[base]] @ P[prow[pid]]):.3f}  {title}")

## 7. In the project

```sh
match block --blocking embed-nn
match eval --run embed
match eval --run p-embed
```

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [A language model for the unsure pairs](https://learning.nextia-ai.com/courses/record-matching/m06/a-language-model-for-the-unsure-pairs/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/record-matching/m06/embeddings-for-names-addresses-and-products/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every customer record and every product listing here is synthetic: our code made them from invented people and products. No real person is in this data.